## 0. 预备

不要看答案，完成下面的程序：
class User:
    # 包含 name 和 email
    # 实现 describe() 方法
    pass


class Admin(User):
    # 比 User 多一个 permissions 属性
    # 实现 show_permissions() 方法
    pass
要求：
1. User 使用构造方法接收 name 和 email
2. describe() 返回用户信息
3. Admin 继承 User
4. permissions 是一个列表
5. 创建一个 Admin 对象，调用两个方法
6. 给参数和返回值添加类型标注
先自己写，不必一次写对。

In [ ]:
class User:
    def __init__(self, name:str, email:str):
        self.name = name
        self.email = email
        
    def describe(self):
        return self.name
    
    pass

class Admin(User, permissions):
    def __init__(self, permissions: list[str]):
        super().__init__()
        self.permissions = permissions
        
    def show_permissions(self):
        print(f"permissions = {self.permissions}")
        
    pass

建立任务骨架:
```
fashion-mnist-classifier/
├── configs/
│   └── default.yaml
├── config.py
├── dataset.py
├── eval.py
├── model.py
|—— README.md
├── requirements.txt
├── train.py
|—— utils.py
```

# 正式开始项目


接下来初始化Git, 创建第一次提交, 例如:
```
chore: initialize project structure
```

### 1. 写好配置文件config.py

In [ ]:
from dataclasses import dataclass


@dataclass    # @dataclass 会自动帮我们生成初始化方法
class TrainConfig:
    batch_size: int = 64
    learning_rate: float = 0.001
    epochs: int = 5
    device: str = "cpu"


def describe_config(config: TrainConfig) -> str:
    print(
        f"batch_size = {self.batch_size}"
        f"learning_rate = {self.learning_rage}"
        f"epochs = {self.epochs}"
        f"device = {self.device}"
    )
    pass

标准库`@dataset`的功能:
- 自动生成构造方法`__init__:
```
from dataclasses import dataclass
""" 两种写法等价 """
@dataclass
class User:
    name: str
    email: str

class User:
    def __init__(self, name: str, email: str):
        self.name = name
        self.email = email
```
- 自动生成字符串表示`__repr__`: 打印对象时会输出清晰的"类名+字段名+字段值"格式, 方便调试
- 自动生成相等比较`__eq__`: 可以直接用`==`比较两个实例是否相等


## 2. 练习模块的导入

在`train.py`中导入配置:

In [ ]:
from config import TrainConfig, describe_config


def main() -> None:
    config = TrainConfig()
    print(describe_config(config))


if __name__ == "__main__":
    main()
    
""" 
执行结果:

train.py
batch_size = 64
learning_rate = 0.001
epochs = 5
device = cpu

"""


## 3. 给config.py增加三个字段:

In [ ]:
# 数据保存目录；后续代码会将它解析为相对于项目目录的路径。
data_dir: str = "data"

# 从官方训练集中取出 6000 张，作为验证集。
val_size: int = 6000

# 固定随机划分使用的种子，方便重复获得相同的数据划分。
seed: int = 42

本项目采用以下划分: 
| 数据部分 | 数量 | 用途 |
|---|---:|---|
| 训练集 | 54,000 | 更新模型参数 |
| 验证集 | 6,000 | 训练过程中比较模型、选择最佳 checkpoint |
| 测试集 | 10,000 | 最后独立评估 |


## 4. 编写数据模块dataset.py

In [ ]:
"""
文件作用: 下载并读取 FashionMNIST, 划分训练集与验证集; 为训练、验证和测试分别按批次读取数据的DataLoader
"""

from pathlib import Path                    # Path 提供跨平台的路径拼接和解析功能
import torch                                # Generator 用于控制数据划分时使用的随机数

# DataLoader 将样本组成批次; random_split 划分数据集
from torch.utils.data import DataLoader, random_split

# FashionMNIST 封装数据下载与读取; ToTensor 将图像转换为张量 
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor

from config import TrainConfig


# tuple[...] 表示返回一个元组, 其中依次放置三个DataLoader.
def build_dataloaders(config: TrainConfig,) -> tuple[DataLoader, DataLoader, DataLoader]:
    """ 返回训练、验证和测试数据的加载器 """

    # __file__ 是当前文件路径; resolve() 得到绝对路径
    # parent 取得文件所在目录: Path 的 / 运算符, 用于拼接路径
    project_dir = Path(__file__).resolve().parent
    data_dir = project_dir / config.data_dir

    
    transform = ToTensor()             # 将灰度图转为[1, 28, 28]的浮点张量, 像素缩放到[0, 1]

    full_train_dataset = FashionMNIST(
        root = str(dat_dir),
        train = True,                   # train = True 读取官方数据集
        transform = transform,
        download = True,                # download = True 在数据缺失时下载, 已存在时直接使用
    )

    if not 0 < config.val_size < len(full_train_dataset):
        raise ValueError("val_size 必须大于0, 且小于训练样本总数")
    
    # 根据配置计算训练大小, 而不是在这里写死54000
    train_size = len(full_train_dataset) - config.val_size

    # manual_seed 固定这个生成器的随机种子, 使划分可以重复
    split_generator = torch.Generator().manual_seed(config.seed)

    # random_split 返回两个子集, 左侧两个变量分别接收它们
    # 两个子集使用不同的样本索引, 不会互相重叠
    train_dataset, val_dataset = random_split(
        full_train_dataset,
        [train_size, config.val_size],
        generator=split_generator,
    )

    test_dataset = FashionMNIST(
        root = str(data_dir),
        train = False,              # train=False 读取官方测试集, 留给最后的独立评估
        transform = transform,
        download = True,
    )

    # shuffle = True 在每次便利训练集时打乱样本顺序
    # num_workers = 0 表示由当前进程加载数据, 先采用简单的运行方式
    train_loader = DataLoader(
        train_dataset,
        batch_size = config.batch_size,
        shuffle=True,
        num_workers=0,
    )

    # 验证与测试不更新参数, 这里固定顺序读取
    val_loader = DataLoader(
        val_dataset,
        batch_size = config.batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size = config_batch_size,
        shuffle = False,
        num_workers=0,
    )

    # 按训练、验证和测试的顺序返回 调用处也按这个顺序接收
    return train_loader,, val_loader, test_loader

## 5. 更新train.py文件

1️⃣.最初的`train.py`只有`main()`, 流程是:

创建配置

    ↓

创建训练、验证、测试加载器

    ↓

输出三个数据集的样本数量

    ↓

取一个训练批次，检查形状和类型

In [ ]:
""" 
组织训练程序的执行流程, 当前阶段加载配置与数据, 检查数据集大小和一个批次的形状
"""

# 导入项目内部的配置模块和数据加载模块
from config import TrainConfig, describe_config
from dataset import build_dataloaders

def main() -> None:
    """加载数据, 并输出检查信息"""
    
    # 创建配置, 沿用已有的参数摘要输出
    config = TrainConfig()
    print(describe_config(config))
    
    # 元组解包: 依次接收函数返回的三个数据加载器.
    train_loader, val_loader, test_loader = build_dataloaders(config)
    
    # loader.dataset 是加载器使用的数据集
    # len(loader.dataset)表示样本数量; len(loader)表示批次数量
    print(f"训练集样本数: {len(train_loader.dataset)}")
    print(f"验证集样本数: {len(val_loader.dataset)}")
    print(f"测试集样本数: {len(test_loader.dataset)}")
    
    # iter() 创建迭代器, next()从中取一个批次
    # 这个批次包含图像张量和标签张量, 用两个变量分别接收
    images, labels = next(iter(train_loader))
    
    # shape 是张量各维度的大小, dtype是张量中元素的数据类型
    print(f"图像形状: {image.shape}")
    print(f"标签形状: {labels.shape}")
    print(f"图像类型: {images.dtype}")
    print(f"标签类型: {labels.dtype}")
    
# 直接运行此文件时执行入口函数
if __name__ == "__main___":
    main()


2️⃣接入模型: 确认图像能够变成分类分数



In [ ]:
# 引入自己定义的分类模型。
from model import SimpleClassifier

在取得一个批次后, 增加:

In [ ]:
# 创建模型：784 个输入像素，10 个输出类别。
model = SimpleClassifier(input_size=28 * 28, num_classes=10)

# 模型和数据使用同一计算设备。
model = model.to(config.device)
images = images.to(config.device)
labels = labels.to(config.device)

# 前向传播，得到每张图的 10 个类别分数。
logits = model(images)

数据流扩展为:

图像 [64, 1, 28, 28]

  ↓

模型内部展平与全连接计算

  ↓

原始类别分数 [64, 10]

这一阶段只检查模型的计算流程和输出形状, 模型参数仍然没有更新

3️⃣. 加入损失函数和优化器: 完成一次参数更新

In [ ]:
# nn 提供损失函数，optim 提供优化器。
from torch import nn, optim

先创建两个对象:

In [ ]:
# 计算分类预测与真实标签之间的损失。
criterion = nn.CrossEntropyLoss()

# 管理模型参数，并按照学习率执行参数更新。
optimizer = optim.Adam(
    model.parameters(),
    lr=config.learning_rate,
)

将原来的单词前向传播扩展成完整的训练步骤

In [ ]:
# 设置训练模式。
model.train()

# 清除已有梯度。
optimizer.zero_grad()

# 前向传播，计算预测分数。
logits = model(images)

# 根据预测分数与真实类别计算损失。
loss = criterion(logits, labels)

# 根据损失计算参数梯度。
loss.backward()

# 根据梯度更新参数。
optimizer.step()

这里新增了三个核心职责:
| 对象或操作 | 职责 |
|---|---|
| `criterion` | 衡量预测与标签之间的损失 |
| `loss.backward()` | 计算各参数的梯度 |
| `optimizer.step()` | 使用梯度更新参数 |

4️⃣. 抽出`train_one_epoch()`: 从一批扩展到整个训练集


原来的训练步骤直接写在`main()`中, 只处理:

In [ ]:
# 只读取一个批次。
images, labels = next(iter(train_loader))

后来新增`train_one_epoch()`, 把训练步骤移到函数内部, 并改成:

In [ ]:
# 依次处理训练加载器提供的全部批次。
for images, labels in loader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    logits = model(images)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()

变化重点在于:
- `main()`负责准备配置、数据、模型和优化器
- `train_one_epoch`负责完整遍历一次训练集
- 设备移动和参数更新放入批次循环, 确保每一批都执行
- 新增了`DataLoader`的导入, 用于函数参数的类型标注
- 一个batch更新一次参数, 一个epoch遍历全部训练样本


5️⃣. 在单轮训练中增加统计: 让训练效果可观察

为了知道整轮训练的表现, 在循环前初始化:

In [ ]:
# 记录本轮累计损失、正确预测数和样本总数。
total_loss = 0.0
total_correct = 0
total_samples = 0

每处理一批, 就累计:


In [ ]:
# 取得当前批次的实际样本数。
batch_size = images.size(0)

# 将批次平均损失转换为批次损失总和。
total_loss += loss.item() * batch_size

# 最大类别分数的位置就是预测类别。
predictions = logits.argmax(dim=1)

# 累计正确预测数量和样本数量。
total_correct += (predictions == labels).sum().item()
total_samples += batch_size

遍历结束后返回:

In [ ]:
# 根据全部样本计算整轮指标。
average_loss = total_loss / total_samples
accuracy = total_correct / total_samples
return average_loss, accuracy

""" 
整轮平均损失 = 各批次“平均损失 × 样本数”的总和 ÷ 总样本数

整轮准确率 = 正确预测总数 ÷ 总样本数
"""

6️⃣. 在`main()`中增加epoch循环, 连续训练多轮

In [ ]:
# 重复训练配置中指定的轮数。
for epoch in range(1, config.epochs + 1):
    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=config.device,
    )

    # 输出当前轮次与本轮训练指标。
    print(
        f"Epoch {epoch}/{config.epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.2%}"
    )

得到了新版的`train.py`文件(但仍不是最终版)

In [ ]:
"""
组织配置, 数据加载和模型创建
执行多轮训练， 并输出每轮训练集平均损失与准确率
"""

# 导入项目内部的配置模块和数据加载模块。
import torch


from torch import nn, optim
from config import TrainConfig, describe_config
from dataset import build_dataloaders
from model import SimpleClassifier
from torch.utils.data import DataLoader

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: str,
) -> tuple[float, float]:
    """完整遍历一次训练集，更新参数并汇总训练指标"""

    # 设置训练模式, 并初始化本轮统计量
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    # 每次循环从加载器取出一批图像及其标签
    for images, labels in loader:
        # 将当前批次移动到与模型相同的设备
        images = images.to(device)
        labels = labels.to(device)
        
        # 清除上一批次的梯度, 随后计算本批次的预测与损失
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        
        # 根据损失梯度, 再由优化器更新参数
        loss.backward()
        optimizer.step()
        
        # size(0) 读取第0维, 即当前批次实际包含的样本数.
        batch_size = images.size(0)
        
        # loss 默认是批次平均值, 乘样本数得到批次损失总和
        # 最后除以总样本数, 得到按样本加权的整轮平均损失
        total_loss += loss.item() * batch_size
        
        # argmax(dim=1) 对每张图的10个分数取最大值所在的类别编号
        predictions = logits.argmax(dim=1)
        
        # 逐个比较预测和标签
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size
        
    # 准确率返回0~1的比例, 输出时再格式化为百分比
    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return average_loss, accuracy

def main() -> None:
    """加载数据，并输出检查信息。"""

    # 创建配置，沿用已有的参数摘要输出。
    config = TrainConfig()
    print(describe_config(config))

    # 元组解包：依次接收函数返回的三个数据加载器。
    train_loader, val_loader, test_loader = build_dataloaders(config)

    # loader.dataset 是加载器使用的数据集。
    # len(loader.dataset) 表示样本数量；len(loader) 表示批次数量。
    print(f"训练集样本数: {len(train_loader.dataset)}")
    print(f"验证集样本数: {len(val_loader.dataset)}")
    print(f"测试集样本数: {len(test_loader.dataset)}")

    # 模型、损失函数和优化器在多轮训练开始前创建一次， 后续各轮
    # 后续各轮延续已有的模型参数与优化器状态
    model = SimpleClassifier(input_size=28*28, num_classes=10)
    model = model.to(config.device)
    
    criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    
    print(model.describe())
    
    for epoch in range(1, config.epochs + 1):
        # 每次调用完整遍历训练集, 并接收本轮的两个训练指标
        train_loss, train_acc = train_one_epoch(
            model = model,
            loader = train_loader,
            criterion = criterion,
            optimizer = optimizer,
            device = config.device,
        )
        
        print(
            f"Epoch {epoch}/{config.epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.2%}"
        )

# 直接运行此文件时执行入口函数。
if __name__ == "__main__":
    main()

In [ ]:
""" 
运行结果示例:

epochs = 5
device = cpu
训练集样本数: 54000
验证集样本数: 6000
测试集样本数: 10000
Classifier: input_size = 784, num_classes = 10
Epoch 1/5 | train_loss=0.5553 | train_acc=80.90%
Epoch 2/5 | train_loss=0.4030 | train_acc=85.66%
Epoch 3/5 | train_loss=0.3615 | train_acc=87.03%
Epoch 4/5 | train_loss=0.3332 | train_acc=87.90%
Epoch 5/5 | train_loss=0.3119 | train_acc=88.73%
"""

## 6. 编写`eval.py`

In [ ]:
"""
文件作用：提供模型评估函数，计算数据集上的平均损失与准确率。
训练时用于验证，后续也可复用于独立测试。
"""

# torch 提供关闭梯度计算的功能；nn 用于标注模型和损失函数。
import torch
from torch import nn
from torch.utils.data import DataLoader


# 装饰器：在本函数执行期间关闭梯度记录，减少评估的内存开销。
# 函数结束后，恢复调用前的梯度记录状态。
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
) -> tuple[float, float]:
    """遍历评估数据，返回平均损失与准确率，不更新模型参数。"""

    # 设置评估模式，调整 Dropout、BatchNorm 等层的行为。
    model.eval()

    # 初始化整个评估数据集的统计量。
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    # 逐批读取图像与标签。
    for images, labels in loader:
        # 将数据移动到模型所在设备。
        images = images.to(device)
        labels = labels.to(device)

        # 使用当前模型计算预测与损失。
        logits = model(images)
        loss = criterion(logits, labels)

        # 按实际样本数累计损失，兼顾最后一个不满批次。
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size

        # 取最大分数对应的类别编号，再统计正确预测数量。
        predictions = logits.argmax(dim=1)
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

    # 汇总整个数据集的平均损失与准确率。
    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return average_loss, accuracy

写完了`eval.py`后, 在`train.py`的导入区域添加:

In [ ]:
# 导入评估函数，用于每轮训练结束后的验证。
from eval import evaluate

将`main()`中的整个`for epoch ....`替换为:

In [ ]:
    # 每轮先更新模型参数，再用验证集评估当前模型。
    for epoch in range(1, config.epochs + 1):
        # 训练函数内部调用 model.train()，切回训练模式。
        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=config.device,
        )

        # 主要改动部分, 相较于之前版本的train.py, 该版本添加了val_loss,val_acc两个统计量来反映模型在测试集上的表现
        # 使用独立的验证集计算指标，整个验证过程不更新参数。
        val_loss, val_acc = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=config.device,
        )

        # 同时输出训练与验证结果，便于观察它们随轮次的变化。
        print(
            f"Epoch {epoch}/{config.epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.2%} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.2%}"
        )

运行:
```
python -X utf8 train.py
```
结果:
```
batch_size = 64
learning_rate = 0.001
epochs = 5
device = cpu
训练集样本数: 54000
验证集样本数: 6000
测试集样本数: 10000
Classifier: input_size = 784, num_classes = 10
Epoch 1/5 | train_loss=0.5625 | train_acc=80.65% | val_loss=0.4518 | val_acc=84.22%
Epoch 2/5 | train_loss=0.4068 | train_acc=85.62% | val_loss=0.4129 | val_acc=85.75%
Epoch 3/5 | train_loss=0.3684 | train_acc=86.88% | val_loss=0.3735 | val_acc=86.57%
Epoch 4/5 | train_loss=0.3393 | train_acc=87.71% | val_loss=0.3514 | val_acc=87.33%
Epoch 5/5 | train_loss=0.3191 | train_acc=88.40% | val_loss=0.3453 | val_acc=87.73%
```

由以上的运行结果, 可以得出结论:
1. 训练与验证均已跑通
2. 这次验证损失从`0.4518`降到`0.3453`, 最佳验证准确率为第五轮的87.73%

接下来, **保存验证准确率最高的模型**. 以后最佳结果可能出现在中间某轮, 因此需要边训练边保存

In [ ]:
1️⃣. 在训练入口train.py添加导入

In [ ]:
# asdict 将 dataclass 配置对象转换为普通字典，方便保存。
from dataclasses import asdict

# Path 用于计算保存目录和文件路径。
from pathlib import Path

2️⃣. 在epoch循环前准备保存路径
放在`print(model.describe())`后, `for epoch ...`前

In [ ]:
    # 根据当前脚本的位置确定项目目录，避免依赖终端所在目录。
    project_dir = Path(__file__).resolve().parent
    checkpoint_dir = project_dir / "checkpoints"

    # parents=True 允许创建缺失的父目录。
    # exist_ok=True 表示目录已经存在时不报错。
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    # 每次验证准确率提高时，覆盖这个文件。
    checkpoint_path = checkpoint_dir / "best.pt"

    # 准确率范围是 0～1，初始化为 -1，确保第一轮也能保存。
    best_val_acc = -1.0
    best_epoch = 0

3️⃣. 在每轮输出之后判断是否保存
放在现有的指标`print(...)`的后面, 仍然位于`for epoch`循环内部

In [ ]:
        # 只根据验证集准确率选择最佳模型；相同时保留较早的一轮。
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch

            # 将恢复模型所需的状态和实验信息组织为字典。
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_acc": val_acc,
                "config": asdict(config),
            }

            # 将当前状态立即写入磁盘，保留这一轮的参数。
            torch.save(checkpoint, checkpoint_path)

            print(
                f"已保存最佳模型：epoch={epoch} | "
                f"val_acc={val_acc:.2%}"
            )

4️⃣. 在训练循环结束后, 输出摘要
下面两条输出缩进为4个空格, 位于`main()`内, `for`循环外

In [ ]:
    # 全部训练完成后，报告最终保留的是哪一轮。
    print(
        f"训练结束，最佳轮次={best_epoch}，"
        f"最佳验证准确率={best_val_acc:.2%}"
    )
    print(f"模型保存位置：{checkpoint_path}")

现在, `main()`的后半段应该是:
```
准备保存路径、初始化最佳指标
  ↓
for：逐轮训练
    ├── 训练
    ├── 验证
    ├── 输出本轮指标
    └── 如果验证准确率提高：保存 best.pt
  ↓
输出最佳轮次与保存位置
```


运行:
```
# 重新训练，并在过程中保存最佳模型。
python -X utf8 train.py                 
```
结果:
```
batch_size = 64
learning_rate = 0.001
epochs = 5
device = cpu
训练集样本数: 54000
验证集样本数: 6000
测试集样本数: 10000
Classifier: input_size = 784, num_classes = 10
Epoch 1/5 | train_loss=0.5604 | train_acc=80.70% | val_loss=0.4813 | val_acc=82.92%
已保存最佳模型：epoch=1 | val_acc=82.92%
Epoch 2/5 | train_loss=0.4110 | train_acc=85.45% | val_loss=0.4028 | val_acc=85.53%
已保存最佳模型：epoch=2 | val_acc=85.53%
Epoch 3/5 | train_loss=0.3652 | train_acc=87.03% | val_loss=0.3579 | val_acc=87.22%
已保存最佳模型：epoch=3 | val_acc=87.22%
Epoch 4/5 | train_loss=0.3394 | train_acc=87.75% | val_loss=0.3427 | val_acc=87.83%
已保存最佳模型：epoch=4 | val_acc=87.83%
Epoch 5/5 | train_loss=0.3184 | train_acc=88.44% | val_loss=0.3450 | val_acc=87.20%
训练结束，最佳轮次=4，最佳验证准确率=87.83%
模型保存位置：D:\code\26fall_AiLearning\projects\FashionMNIST_review\checkpoints\best.pt
```


## 7.编写日志工具`utils.py`, 并接入train.py

1️⃣. 编写日志工具, 将以下代码写入空文件utils.py

In [ ]:
"""
提供项目通用工具
当前负责创建日志记录器, 将运行信息同时输出到终端和日志文件
"""

# logging 是 Python 标准库提供的日志模块，无需额外安装。
import logging
from pathlib import Path


def setup_logger(log_dir: Path) -> logging.Logger:
    """创建日志目录，配置终端与文件输出，返回日志记录器。"""

    # 创建日志目录；目录已经存在时继续使用。
    log_dir.mkdir(parents=True, exist_ok=True)

    # 获取指定名称的记录器，INFO 级别用于记录正常运行信息。
    logger = logging.getLogger("fashionmnist")
    logger.setLevel(logging.INFO)

    # 禁止继续向父记录器传递，避免同一条信息重复输出。
    logger.propagate = False

    # 如果重复调用本函数，先关闭旧的输出处理器。
    # [:] 创建列表副本，避免在遍历原列表时修改它。
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
        handler.close()

    # 定义每条日志的格式：时间、级别、具体消息。
    # %(...)s 是 logging 使用的占位符，由日志模块填入对应字段。
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    # 文件处理器：以 UTF-8 编码保存中文日志。
    # mode="a" 表示追加，保留之前运行留下的记录。
    file_handler = logging.FileHandler(
        filename=log_dir / "train.log",
        mode="a",
        encoding="utf-8",
    )
    file_handler.setFormatter(formatter)

    # 终端处理器：将相同的信息显示在终端。
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)

    # 一个记录器连接两个处理器，实现同时输出。
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    return logger

三个对象的关系可以记为
```
logger：接收消息
  ├── FileHandler：写入文件
  └── StreamHandler：显示到终端

Formatter：决定两种输出的文字格式
```

2️⃣. 在训练入口初始化日志

在`train.py`的导入区添加:

In [ ]:
# 导入日志配置函数。
from utils import setup_logger

将`main()`开头创建和输出配置的部分改为:


In [ ]:
    # 创建训练配置。
    config = TrainConfig()

    # 通过脚本路径确定项目位置，并创建日志记录器。
    project_dir = Path(__file__).resolve().parent
    logger = setup_logger(project_dir / "logs")

    # 标记一次新的运行，方便区分追加文件中的不同实验。
    logger.info("=" * 50)
    logger.info("开始新的训练")
    logger.info(describe_config(config))

下面准备 checkpoint 的位置原来还有：

In [ ]:
    project_dir = Path(__file__).resolve().parent

删除那一条重复赋值，保留后面的：

In [ ]:
    checkpoint_dir = project_dir / "checkpoints"

3️⃣. 将`main()`中的输出改为日志
将其中所有`print(...)`改为`logger.info(...)`, 括号内原有内容保持不变

例如, 样本数量改为:

In [ ]:
    # 同时记录到终端和日志文件。
    logger.info(f"训练集样本数: {len(train_loader.dataset)}")
    logger.info(f"验证集样本数: {len(val_loader.dataset)}")
    logger.info(f"测试集样本数: {len(test_loader.dataset)}")

每轮指标改为:

In [ ]:
        # 保留原来的指标格式，改由日志记录器输出。
        logger.info(
            f"Epoch {epoch}/{config.epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.2%} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.2%}"
        )

在终端运行并检查日志:

In [ ]:


# 按 UTF-8 编码读取日志文件的最后 20 行。
Get-Content -LiteralPath '.\logs\train.log' -Encoding UTF8 -Tail 20


运行:
```
# 运行训练，同时生成日志。
python -X utf8 train.py
```
结果:
```
2026-10-22 14:56:47 | INFO | ==================================================
2026-10-22 14:56:47 | INFO | 开始新的训练
2026-10-22 14:56:47 | INFO | batch_size = 64
learning_rate = 0.001
epochs = 5
device = cpu
2026-10-22 14:56:47 | INFO | 训练集样本数: 54000
2026-10-22 14:56:47 | INFO | 验证集样本数: 6000
2026-10-22 14:56:47 | INFO | 测试集样本数: 10000
Classifier: input_size = 784, num_classes = 10
2026-10-22 14:56:51 | INFO | Epoch 1/5 | train_loss=0.5636 | train_acc=80.46% | val_loss=0.4674 | val_acc=83.13%
已保存最佳模型：epoch=1 | val_acc=83.13%
2026-10-22 14:56:56 | INFO | Epoch 2/5 | train_loss=0.4118 | train_acc=85.33% | val_loss=0.4065 | val_acc=85.95%
已保存最佳模型：epoch=2 | val_acc=85.95%
2026-10-22 14:57:00 | INFO | Epoch 3/5 | train_loss=0.3673 | train_acc=86.89% | val_loss=0.3973 | val_acc=85.90%
2026-10-22 14:57:05 | INFO | Epoch 4/5 | train_loss=0.3409 | train_acc=87.64% | val_loss=0.3736 | val_acc=86.63%
已保存最佳模型：epoch=4 | val_acc=86.63%
2026-10-22 14:57:09 | INFO | Epoch 5/5 | train_loss=0.3201 | train_acc=88.39% | val_loss=0.3659 | val_acc=86.80%
已保存最佳模型：epoch=5 | val_acc=86.80%
训练结束，最佳轮次=5，最佳验证准确率=86.80%
模型保存位置：D:\code\26fall_AiLearning\projects\FashionMNIST_review\checkpoints\best.pt
```

继续运行:
```
2026-10-22 14:56:47 | INFO | ==================================================
2026-10-22 14:56:47 | INFO | 开始新的训练
2026-10-22 14:56:47 | INFO | batch_size = 64
learning_rate = 0.001
epochs = 5
device = cpu
2026-10-22 14:56:47 | INFO | 训练集样本数: 54000
2026-10-22 14:56:47 | INFO | 验证集样本数: 6000
2026-10-22 14:56:47 | INFO | 测试集样本数: 10000
Classifier: input_size = 784, num_classes = 10
2026-10-22 14:56:51 | INFO | Epoch 1/5 | train_loss=0.5636 | train_acc=80.46% | val_loss=0.4674 | val_acc=83.13%
已保存最佳模型：epoch=1 | val_acc=83.13%
2026-10-22 14:56:56 | INFO | Epoch 2/5 | train_loss=0.4118 | train_acc=85.33% | val_loss=0.4065 | val_acc=85.95%
已保存最佳模型：epoch=2 | val_acc=85.95%
2026-10-22 14:57:00 | INFO | Epoch 3/5 | train_loss=0.3673 | train_acc=86.89% | val_loss=0.3973 | val_acc=85.90%
2026-10-22 14:57:05 | INFO | Epoch 4/5 | train_loss=0.3409 | train_acc=87.64% | val_loss=0.3736 | val_acc=86.63%
已保存最佳模型：epoch=4 | val_acc=86.63%
2026-10-22 14:57:09 | INFO | Epoch 5/5 | train_loss=0.3201 | train_acc=88.39% | val_loss=0.3659 | val_acc=86.80%
已保存最佳模型：epoch=5 | val_acc=86.80%
训练结束，最佳轮次=5，最佳验证准确率=86.80%
模型保存位置：D:\code\26fall_AiLearning\projects\FashionMNIST_review\checkpoints\best.pt
```



## 7. 为`eval.py`添加独立测试入口
现在进入最后一个主要功能: 从磁盘取出最佳模型, 评估10000张测试图像

在`eval.py`顶部添加导入:

In [2]:
# Path 用于定位项目与 checkpoint 文件。
from pathlib import Path

# 根据保存的配置恢复数据加载方式，并重建模型。
from config import TrainConfig
from dataset import build_dataloaders
from model import SimpleClassifier

保留现有的`evaluate()`, 在它的后面添加:

In [3]:
def main() -> None:
    """从磁盘恢复最佳模型，独立计算测试集指标。"""

    # 根据脚本位置定位模型文件，避免依赖终端所在目录。
    project_dir = Path(__file__).resolve().parent
    checkpoint_path = project_dir / "checkpoints" / "best.pt"

    # map_location="cpu" 将保存的张量加载到 CPU。
    # weights_only=True 使用适合张量和基础数据类型的受限加载方式。
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )

    # ** 将字典展开为关键字参数，例如 batch_size=64。
    # 使用训练时保存的配置，并明确在 CPU 上执行独立评估。
    config = TrainConfig(**checkpoint["config"])
    config.device = "cpu"

    # 先重建与训练时一致的网络结构，再载入训练好的参数。
    model = SimpleClassifier(input_size=28 * 28, num_classes=10)
    model = model.to(config.device)

    # strict=True 要求保存的参数名称与当前模型完全匹配。
    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    # 复用数据模块，只使用返回的第三个加载器，即测试集。
    # 下划线表示这里不使用对应的训练和验证加载器。
    _, _, test_loader = build_dataloaders(config)
    criterion = nn.CrossEntropyLoss()

    # 复用评估函数；其中已有评估模式与关闭梯度记录的设置。
    test_loss, test_acc = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=config.device,
    )

    # 输出模型来源及最终测试结果。
    print(f"模型文件：{checkpoint_path}")
    print(f"保存轮次：{checkpoint['epoch']}")
    print(f"保存的验证准确率：{checkpoint['val_acc']:.2%}")
    print(f"测试集样本数：{len(test_loader.dataset)}")
    print(f"测试损失：{test_loss:.4f}")
    print(f"测试准确率：{test_acc:.2%}")


# 直接运行 eval.py 才执行独立测试。
# train.py 导入 evaluate 时，不会执行这里的 main()。
if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined

这里最关键的区别是:
```
训练：创建模型 → 更新参数 → 保存 checkpoint

独立测试：创建相同结构 → 加载保存的参数 → 计算测试指标
```

在终端运行:
```
# 使用当前已有的`best.pt`进行测试, 无需重新训练
python -X utf8 eval.py
```
运行结果:
```
模型文件：D:\code\26fall_AiLearning\projects\FashionMNIST_review\checkpoints\best.pt
保存轮次：5
保存的验证准确率：86.80%
测试集样本数：10000
测试损失：0.3858
测试准确率：86.09%
```


#### 独立运行eval.py进行测试和运行train.py完成训练加测试, 二者区别在哪
两者复用同一个`evaluate()`函数, 但模型来源、数据和执行时机不同
| 对比项 | 运行 `train.py` | 独立运行 `eval.py` |
|---|---|---|
| 模型来源 | 当前进程中正在训练的模型 | 从 `best.pt` 恢复的模型 |
| 是否训练 | 每轮先更新模型参数 | 直接加载参数进行评估 |
| 评估数据 | 验证集，6,000 张 | 测试集，10,000 张 |
| 执行时机 | 每轮训练结束后 | 训练完成后单独执行 |
| 结果用途 | 比较各轮表现，保存最佳模型 | 报告最终测试表现 |